#36615 - Final Project Part 3 - Delay Prediction Spark ML
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

## Overview and modeling goal

In this notebook, we use Spark ML to predict whether a flight will be departure delayed, where a delay is defined as either `DEP_DELAY` > 0 or the flight being cancelled. We train a Support Vector Machine (SVM) classifier on the feature-engineered dataset created in Part 2, evaluate its performance on a held-out validation set, and compare it to a simple baseline that always predicts “no delay.”

Because our engineered features include rolling, time-dependent signals (e.g., prior-hour weather delay rates and trailing 7-day carrier/route delay rates), we avoid a fully random split. Instead, we split chronologically so that training data comes from “earlier” flights and evaluation data comes from “later” flights, which better reflects real-world prediction.

We implement several optimizations:

1. **Quantile-based time split**: We create a single numeric “time rank” column and use approxQuantile to find the 60th and 80th percentile cutoffs. This avoids a full global sort and expensive window functions while still producing an approximately 60/20/20 chronological split.

2. **Cache + materialize**: We cache train/test/validation splits and transformed feature matrices so Spark does not recompute them for each model fit or evaluation.

3. **Fit preprocessing on train only**: We fit the feature encoding (RFormula) and scaling (StandardScaler) using only training data to prevent leakage.

4. **Tune on a sample first**: We tune hyperparameters on a small sample of the training set to reduce runtime, then fit the best configuration on the full training data.

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import RFormula, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import LinearSVC
from pyspark.ml import Pipeline

import pandas as pd


#### 1. Load feature-engineered data

In [0]:
df = spark.table("lsd_2026.default.polyhymnia_feature_engineered")

print(f"Rows: {df.count():,}")
df.printSchema()

#### 2. Define target label: departure delayed or not

In [0]:
df = df.withColumn(
    "dep_delayed",
    F.when((F.col("DEP_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)

df.groupBy("dep_delayed").count().show()

We see that about 37% of flights in the dataset (over the decade) were delayed.

#### 3. Train / Test / Validation split strategy

Because we used rolling/time-based features (previous hour, past 7 days), we will not perform a fully random split. Instead, we will use the earliest 60% as the training set and next 20% as test set (for model selection / tuning). Final 20% will be used as final validation (held out).


In [0]:
import pyspark.sql.functions as F

# create a numeric representation of time for splitting
# combining date and time into a single numeric value
df = df.withColumn("time_rank_col", 
                   F.unix_timestamp(F.to_date("FL_DATE", "yyyy-MM-dd")) + (F.col("CRS_DEP_TIME").cast("int") * 60))

# find the 60th and 80th percentile values
# 0.001 is the relative error: lower is more accurate but slower
split_points = df.stat.approxQuantile("time_rank_col", [0.6, 0.8], 0.001)

train_threshold = split_points[0]
test_threshold = split_points[1]

# perform the split
train = df.filter(F.col("time_rank_col") <= train_threshold)
test  = df.filter((F.col("time_rank_col") > train_threshold) & (F.col("time_rank_col") <= test_threshold))
valid = df.filter(F.col("time_rank_col") > test_threshold)

# cache the splits so Spark doesn't re-calculate them for every model in the grid
train.cache()
test.cache()

## Support Vector Machine

We now fit a support vector machine classifier to the processed data. Because SVM optimization depends on feature scale, we include StandardScaler so that no single feature dominates due to having a larger numeric range.

#### 4. RFormula and StandardScaler Preprocessing

In [0]:
formula_str = """
dep_delayed ~ 
    OP_CARRIER + 
    CRS_DEP_TIME + CRS_ARR_TIME + CRS_ELAPSED_TIME + DISTANCE +
    day_of_week + hour_of_day +
    crs_dep_time_bucket + arr_time_bucket +
    rate_weather_delay_dep + rate_weather_delay_arr +
    dep_traffic_z_score + carrier_delay_rate_7d + route_delay_rate_7d
"""

# RFormula 
rf_stage = RFormula(
    formula=formula_str,
    featuresCol="raw_features",
    labelCol="label",
    handleInvalid="skip"
)

rf_model = rf_stage.fit(train)

train_rf = rf_model.transform(train)
test_rf  = rf_model.transform(test)
valid_rf = rf_model.transform(valid)

# StandardScaler
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_rf)

train_fe = scaler_model.transform(train_rf).select("label", "features")
test_fe  = scaler_model.transform(test_rf).select("label", "features").cache()
valid_fe = scaler_model.transform(valid_rf).select("label", "features").cache()

#### 5. Tune, fit, and evaluate SVM

In [0]:

# LinearSVC Tuning 
train_sample_fe = train_fe.sample(withReplacement=False, fraction=0.10).cache()

# Define Linear Support Vector Machine
lsvc = LinearSVC(labelCol="label", featuresCol="features")

# Hyperparameters: regParam (Regularization) and maximum iterations
reg_params = [0.01, 0.1]
iter_params = [10, 20]

# Set up evaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

best_f1 = -1.0
best_params = {}

print("Starting SVM Parameter Tuning on Train Sample...")
for reg in reg_params:
    for max_i in iter_params:
        lsvc_curr = lsvc.copy({lsvc.regParam: reg, lsvc.maxIter: max_i})
        
        # Fit on sample
        model_temp = lsvc_curr.fit(train_sample_fe)
        
        # Evaluate on Test set
        preds = model_temp.transform(test_fe)
        f1_score = evaluator.evaluate(preds)
        
        print(f"LinearSVC (regParam={reg}, maxIter={max_i}) | Test F1: {f1_score:.4f}")
        
        if f1_score > best_f1:
            best_f1 = f1_score
            best_params = {'regParam': reg, 'maxIter': max_i}

print(f"\nBest Parameters Found: {best_params}. Now fitting on FULL Training data...")

# Unpersist sample to free memory
train_sample_fe.unpersist()

# Fit the optimal model on FULL training set
best_lsvc = LinearSVC(labelCol="label", featuresCol="features", 
                      regParam=best_params['regParam'], maxIter=best_params['maxIter'])

best_model = best_lsvc.fit(train_fe)
print("Full SVM model training completed successfully!")


# Evaluate Best Model on Validation Set 

valid_preds = best_model.transform(valid_fe)

# Confusion Matrix using Pandas
cm = (valid_preds
      .groupBy("label", "prediction")
      .count()
      .toPandas())

print("Confusion Matrix (Validation):")
print(cm)

TP = cm[(cm["label"] == 1.0) & (cm["prediction"] == 1.0)]["count"].sum()
TN = cm[(cm["label"] == 0.0) & (cm["prediction"] == 0.0)]["count"].sum()
FP = cm[(cm["label"] == 0.0) & (cm["prediction"] == 1.0)]["count"].sum()
FN = cm[(cm["label"] == 1.0) & (cm["prediction"] == 0.0)]["count"].sum()

TP, TN, FP, FN = map(float, [TP, TN, FP, FN])

accuracy = (TP + TN) / (TP + TN + FP + FN)
tpr = TP / (TP + FN) if (TP + FN) > 0 else 0        
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

valid_f1 = evaluator.evaluate(valid_preds)

print("\nValidation Metrics (LinearSVC):")
print(f"Accuracy:           {accuracy:.4f}")
print(f"F1 Score:           {valid_f1:.4f}")
print(f"TPR/Sensitivity:    {tpr:.4f}")
print(f"False Positive Rate:{fpr:.4f}")
print(f"Specificity:        {specificity:.4f}")


# Metrics for Baseline Model 

baseline = valid_fe.withColumn("prediction", F.lit(0.0))

baseline_cm = (baseline
               .groupBy("label", "prediction")
               .count()
               .toPandas())

print("Baseline Confusion Matrix:")
print(baseline_cm)

B_TP = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_TN = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()
B_FP = baseline_cm[(baseline_cm["label"] == 0.0) & (baseline_cm["prediction"] == 1.0)]["count"].sum()
B_FN = baseline_cm[(baseline_cm["label"] == 1.0) & (baseline_cm["prediction"] == 0.0)]["count"].sum()

B_acc = (B_TP + B_TN) / (B_TP + B_TN + B_FP + B_FN)
B_tpr = B_TP / (B_TP + B_FN) if (B_TP + B_FN) > 0 else 0
B_fpr = B_FP / (B_FP + B_TN) if (B_FP + B_TN) > 0 else 0
B_spec = B_TN / (B_TN + B_FP) if (B_TN + B_FP) > 0 else 0

print("\nBaseline Metrics:")
print(f"Accuracy:           {B_acc:.4f}")
print(f"TPR/Sensitivity:    {B_tpr:.4f}")
print(f"True Negative Rate: {B_spec:.4f}")


# Comparison Summary

print("\n=== Validation Performance Comparison (LinearSVC vs Baseline) ===")
print(f"SVM Accuracy vs Baseline: {accuracy:.4f} vs {B_acc:.4f}")
print(f"SVM TPR vs Baseline:      {tpr:.4f} vs {B_tpr:.4f}")
print(f"SVM FPR vs Baseline:      {fpr:.4f} vs {B_fpr:.4f}")
print(f"SVM Specificity vs Base:  {specificity:.4f} vs {B_spec:.4f}")

The SVM is noticeably more willing than the baseline to predict delays, which is why it achieves a much higher recall. On the validation set, it correctly identifies about 39% of delayed flights, whereas the baseline (which always predicts “no delay”) never catches any delays at all. The downside is that this comes with more false alarms: the SVM incorrectly flags some on-time flights as delayed, reflected in its higher false positive rate and lower specificity compared to the baseline. The model is not simply maximizing accuracy by defaulting to the majority class; instead, it makes a meaningful tradeoff by catching a real portion of delays at the cost of occasionally over-predicting them.